# 04 -- Map-Algebra Foundations

This notebook introduces the eager `Raster` value type: construction,
inspection, arithmetic, comparisons, Boolean logic, validity handling,
grid alignment, units, and numerical policies.  It draws from command-line
examples 18 through 21.

**Run the individual scripts:** `18_map_algebra_basics.py`,
`19_map_algebra_validity.py`, `20_map_algebra_grids.py`,
`21_map_algebra_numerics.py`

## Setup

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError("Cannot locate Lunarscout repository root.")

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))


import lunarscout as ls
import numpy as np

ma = ls.map_algebra

MOON_WKT = """PROJCS["ESRI:103878",GEOGCS["Moon_2000",DATUM["D_Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],PRIMEM["Reference_Meridian",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-90],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1]]"""
MOON_PROJ4 = "+proj=stere +lat_0=-90 +lon_0=0 +k=1 +x_0=0 +y_0=0 +R=1737400 +units=m +no_defs"

DEMO = ls.GeoReference(
    projection_wkt="""PROJCS["ESRI:103878",GEOGCS["Moon_2000",DATUM["D_Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],PRIMEM["Reference_Meridian",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-90],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1]]""",
    projection_proj4="+proj=stere +lat_0=-90 +lon_0=0 +k=1 +x_0=0 +y_0=0 +R=1737400 +units=m +no_defs",
    affine_transform=(-20.0, 10.0, 0.0, 20.0, 0.0, -10.0),
    width=4, height=3,
    pixel_size_x=10.0, pixel_size_y=-10.0,
    nodata=None,
)

print(f"Demo grid: {DEMO.width}x{DEMO.height}, {DEMO.pixel_size_x}m pixels")

---
## 1. Building and Inspecting a Raster

A `Raster` stores ordinary NumPy values together with a validity mask,
georeference, optional units, and an optional name.

In [ ]:
rng = np.random.default_rng(42)
values = rng.uniform(0, 15, (DEMO.height, DEMO.width)).astype(np.float32)
valid = np.ones((DEMO.height, DEMO.width), dtype=bool)
valid[1, 1] = False   # mark one cell invalid

slope = ma.raster(values, DEMO, valid=valid, units="deg", name="slope")

print(slope)
print(f"\n  values dtype:  {slope.values.dtype}")
print(f"  values shape:   {slope.values.shape}")
print(f"  valid count:    {slope.valid.sum()}")
print(f"  units:          {slope.units}")
print(f"  name:           {slope.name}")

In [ ]:
# Inspect values and validity side by side.
print("Values:           Validity:")
for y in range(DEMO.height):
    v_line = "  ".join(f"{slope.values[y, x]:5.1f}" for x in range(DEMO.width))
    m_line = "  ".join(f"{'V' if slope.valid[y, x] else 'X':>5}" for x in range(DEMO.width))
    print(f"{v_line}   {m_line}")

The cell at row 1, column 1 is invalid even though it has a numeric
payload.  Validity is independent of the payload value.

**Try this:** create a `Raster` with `valid` all `True` and print it.
Then create one where the validity mask does not match the value shape
and observe the error.

---
## 2. Local Algebra

Map-algebra operations return new rasters, preserve metadata, and intersect
operand validity.

In [ ]:
elevation = ma.raster(
    np.array([[1100, 1050, 1005,  980],
              [1200, 1150,    0,  950],
              [1300, 1250, 1200, 1150]], dtype=np.float32),
    DEMO, units="m", name="elevation",
)

# Beware: adding metres and degrees raises MapAlgebraUnitError.
try:
    elevation + slope
except ls.MapAlgebraUnitError as e:
    print(f"m + deg -> MapAlgebraUnitError: code={e.code}")

In [ ]:
# Compatible operations: arithmetic, comparisons, clip.
scaled = elevation / 1000.0          # -> dim-less
steep = slope >= 8.0                  # -> bool
clipped = ma.clip(slope, lower=2.0, upper=10.0)

print(f"scaled units:  {scaled.units}")
print(f"steep dtype:   {steep.values.dtype}")
print(f"clipped range: [{clipped.values.min():.1f}, {clipped.values.max():.1f}]")

In [ ]:
# Boolean operations: & | ~  (not 'and'/'or')
sun = elevation >= 1050.0
shadow = ~sun

both = steep & ~shadow
either = steep | ~shadow

print(f"Steep & ~shadow:  {both.values.sum()} cells")
print(f"Steep | ~shadow:  {either.values.sum()} cells")

Python `and` / `or` are rejected.  Use `&`, `|`, and `~` for raster
Boolean logic.

**Try this:** combine three conditions with `&`.  Compute `slope > 10.0`
and `elevation < 1100.0` and intersect them.

---
## 3. Validity, `where`, and `coalesce`

This is the most important validity lesson.  The inputs below contain a
valid zero, an invalid zero payload, and an invalid non-zero payload that
looks plausible.

In [ ]:
# Three values at three pixels.
values_A = np.array([[1.0, 0.0, 99.0]], dtype=np.float32)
valid_A =  np.array([[True, True, False]], dtype=bool)

values_B = np.array([[0.5, 0.5, 0.5]], dtype=np.float32)
valid_B =  np.array([[True, False, False]], dtype=bool)

g1x3 = ls.GeoReference(
    projection_wkt=MOON_WKT, projection_proj4=MOON_PROJ4,
    affine_transform=(0.0, 10.0, 0.0, 0.0, 0.0, -10.0),
    width=3, height=1, pixel_size_x=10.0, pixel_size_y=-10.0,
    nodata=None,
)
A = ma.raster(values_A, g1x3, valid=valid_A, name="A")
B = ma.raster(values_B, g1x3, valid=valid_B, name="B")

print("A: values", A.values, "valid", A.valid)
print("B: values", B.values, "valid", B.valid)

In [ ]:
# Ordinary arithmetic intersects operand validity.
C = A + B
print("A + B:")
print("  values:", C.values, "valid:", C.valid)

# where: requires condition AND selected branch to be valid.
w = ma.where(A < 10.0, A, ma.invalid)
print("\nwhere(A < 10, A, invalid):")
print("  values:", w.values, "valid:", w.valid)

In [ ]:
# coalesce: takes the first valid value.
c = ma.coalesce(A, B)
print("coalesce(A, B):")
print("  values:", c.values, "valid:", c.valid)

c2 = ma.coalesce(B, A)
print("coalesce(B, A):")
print("  values:", c2.values, "valid:", c2.valid)

In [ ]:
# fill_invalid converts missing cells to valid caller values.
filled = ma.fill_invalid(A, -999.0)
print("fill_invalid(A, -999):")
print("  values:", filled.values, "valid:", filled.valid)

Reversing `coalesce` order changes meaning.  Filling with zero also changes
meaning: the cells cease to be missing.  Do not use filling merely to make a
plot look complete.

**Try this:** create a raster where all cells are invalid and then
`coalesce` it with another.  What happens?

---
## 4. Explicit Grid Alignment

Two same-shaped `Raster` values on shifted grids are rejected.

In [ ]:
g2 = ls.GeoReference(
    projection_wkt=MOON_WKT, projection_proj4=MOON_PROJ4,
    affine_transform=(-15.0, 10.0, 0.0, 20.0, 0.0, -10.0),
    width=4, height=3, pixel_size_x=10.0, pixel_size_y=-10.0,
    nodata=None,
)

r1 = ma.raster(values, DEMO, name="on_demo")
r2 = ma.raster(values, g2, name="on_shifted")

try:
    _ = r1 + r2
except ls.MapAlgebraGridError as e:
    print(f"MapAlgebraGridError: code={e.code}")
    print(f"  {e.details}")

In [ ]:
# Align explicitly.
r2_aligned = ma.align(r2, to=DEMO, resampling="bilinear", output_nodata=-9999.0)
result = r1 + r2_aligned   # succeeds
print(f"After alignment: {result.values[0, :2]} ...")

**Try this:** use `ma.row_indices(DEMO)` and `ma.projected_x(DEMO)` to
build coordinate rasters.  Materialise them with `ma.compute()`.

---
## 5. Units and Numerical Policies

Units are tracked through operations.  Numerical policies control overflow,
casting, and non-finite handling.

In [ ]:
# Use a 1-row, 3-column grid for these demos.
g1x3 = ls.GeoReference(
    projection_wkt=MOON_WKT, projection_proj4=MOON_PROJ4,
    affine_transform=(0.0, 10.0, 0.0, 0.0, 0.0, -10.0),
    width=3, height=1, pixel_size_x=10.0, pixel_size_y=-10.0,
    nodata=None,
)
e1 = ma.raster(np.array([[1.0, 2.0, 3.0]], dtype=np.float32),
               g1x3, units="m", name="a")
e2 = ma.raster(np.array([[10.0, 20.0, 30.0]], dtype=np.float32),
               g1x3, units="m", name="b")
e3 = ma.raster(np.array([[1.0, 2.0, 3.0]], dtype=np.float32),
               g1x3, units="deg", name="c")

print(f"m + m  = {(e1 + e2).values}")      # OK
try:
    e1 + e3
except ls.MapAlgebraUnitError as e:
    print(f"m + deg -> MapAlgebraUnitError: {e.code}")

In [ ]:
# Trigonometric functions require angle units.
g1x4 = ls.GeoReference(
    projection_wkt=MOON_WKT, projection_proj4=MOON_PROJ4,
    affine_transform=(0.0, 10.0, 0.0, 0.0, 0.0, -10.0),
    width=4, height=1, pixel_size_x=10.0, pixel_size_y=-10.0,
    nodata=None,
)
angle = ma.raster(np.array([[0.0, 30.0, 45.0, 90.0]], dtype=np.float32),
                  g1x4, units="degrees", name="angle")
s = ma.sin(angle)
print(f"sin([0 30 45 90] degrees): {s.values}")

In [ ]:
# Overflow policies on integer arithmetic.
g1x3 = ls.GeoReference(
    projection_wkt=MOON_WKT, projection_proj4=MOON_PROJ4,
    affine_transform=(0.0, 10.0, 0.0, 0.0, 0.0, -10.0),
    width=3, height=1, pixel_size_x=10.0, pixel_size_y=-10.0,
    nodata=None,
)
tiny = ma.raster(
    np.array([[-100, 50, 120]], dtype=np.int8),
    g1x3, name="int8_values",
)

# raise on overflow.
try:
    tiny + tiny
except (ls.MapAlgebraDTypeError, ls.MapAlgebraOperationError) as e:
    print(f"int8 overflow (raise): code={e.code}")

# promote to a wider dtype.
promoted = ma.add(tiny, tiny, overflow="promote")
print(f"int8 overflow (promote): "
      f"dtype={promoted.values.dtype}, values={promoted.values}")

# wrap (NumPy behaviour).
wrapped = ma.add(tiny, tiny, overflow="wrap")
print(f"int8 overflow (wrap):    "
      f"dtype={wrapped.values.dtype}, values={wrapped.values}")

In [ ]:
# Non-finite policies.
v = ma.raster(np.array([[1.0, 0.0, -1.0, 2.0]], dtype=np.float32), g1x4)

# log of non-positive values.
log_inval = ma.log(v, numeric_errors="invalid")
log_keep  = ma.log(v, numeric_errors="keep")

print("log with numeric_errors='invalid':")
print("  values:", log_inval.values, "valid:", log_inval.valid)
print("log with numeric_errors='keep':")
print("  values:", log_keep.values, "valid:", log_keep.valid)

Under `"keep"`, a NaN payload is still a valid cell.  Under `"invalid"`,
the validity mask carries the failure.  Choose the policy that matches your
scientific intent.

**Try this:** experiment with `ma.cast()` using `casting="safe"` versus
`"unsafe"`.  What happens when you cast a negative float to `uint8`?